# AIT Dataset — Unsupervised Anomaly Detection

Dataset: https://zenodo.org/records/19483937

We explore the **fox** silo first before touching the other 7. The goal of this notebook is just to understand the data — no models yet.

At the end of this notebook we should be able to answer:

1. What hosts exist, and which ones do we *exclude* from training?
2. Which log files actually carry attack signal, and which ones are noise?
3. How do labels map to raw log lines, and do they actually line up?
4. Which attack types are common vs rare? (because the model will not learn the rare ones the same way)
5. What's a reasonable time-window size for aggregation?

Everything below is justification for one of those 5 decisions.

## 1. Setup

### imports

Minimal set. Nothing fancy.

- `pickle`, `random` — for later (saving processed artifacts, seeding).
- `Path` — we do a lot of path work and `os.path` string-joining gets messy fast.
- `yaml` for `dataset.yaml`, `json` for the label files (they're JSONL).
- `numpy`, `pandas` — everything downstream.
- `matplotlib`, `seaborn` — timeline plots later on.

In [ ]:
import pickle
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import yaml, json


### paths

Three globals we need in every cell:

- `FOX` — the silo root. `.resolve()` so relative paths don't break if the notebook is moved.
- `GATHER` — the raw logs. Inside it, `labels/` mirrors the same directory structure exactly (this invariant is what makes the join work).
- `LABELS` — the ground-truth JSONL files.

The last line stops pandas from truncating the wide dataframes we print later. Without it, `label_df` would only show ~20 rows of the ~18 labels and half the log inventory would be cut off.

In [ ]:
FOX = Path("../datasets/ait/fox").resolve()
assert FOX.exists(), f"Download fox first: {FOX}"

GATHER = FOX / "gather"
LABELS = FOX / "labels"
pd.set_option("display.max_rows", 100)

## 2. Data Exploration

### simulation window

Each silo has a specific simulation window. **Only events inside this window are meaningful** for the attack-detection task. Everything outside is testbed setup: cron jobs firing, systemd starting services, apt updating packages. Those events look weird but they're not attacks.

More important: **the label files are already trimmed to the window.** If you filter the raw logs on some other cutoff, or don't filter at all, line numbers won't line up with the labels and your join is garbage. So: know the window, filter to it, every time.

One gotcha: the Zenodo description calls the file `dataset.yml`, but the archive actually ships `dataset.yaml`. Don't trust the docs blindly — check the filesystem.

In [ ]:
with open(FOX / "dataset.yaml") as f:
    meta = yaml.safe_load(f)
print(json.dumps(meta, indent=2, default=str))

### host inventory

30 hosts. They fall into three groups:

**infrastructure:**
- `intranet_server` — the WordPress server. This is the one that gets compromised (webshell + privesc).
- `webserver` — the front-facing webserver
- `mail`, `jonesmorgan_mail`, `miller_mail`, `taylorcruz_mail`, `whittaker_mail` — mail servers. The per-user naming is just how the simulation generates them; treat them all as infrastructure.
- `vpn`, `inet-firewall`, `inet-dns` — network infra
- `cloud_share`, `internal_share` — file shares
- `monitoring` — logstash / monitoring stack

**users:**
- `internal_employee_0..4`, `remote_employee_0..3`, `ext_user_0..6`

**attacker:**
- `attacker_0`

Things that actually matter here:

1. **`attacker_0` must be excluded from normal-behavior training.** Everything on that host is an attack by definition. If you include it, the model learns "attacker_0's pattern is normal".
2. Everyone else generates background noise. That's your training set.
3. `attacker_0/logs/attacks.log` has the exact attack-step timestamps. Useful for sanity-checking your model later — useless for training.

In [ ]:
hosts = sorted(p.name for p in GATHER.iterdir() if p.is_dir())
print(f"{len(hosts)} hosts:")
for h in hosts:
    print(f"  {h}")

### log inventory

Before we parse anything, look at what files exist and how big they are. A 2 GB parquet file is a completely different beast from a 5 MB text file, and the shape of this inventory tells us where to spend effort.

What jumps out in the fox silo:

- `suricata` (2 GB) — network alert logs. Huge. Mostly redundant for our purposes (the firewall already logs the DNS exfil we care about) and slow to parse. **Skip unless you specifically need a network-level detector.**
- `intranet-server` (1.1 GB) — logstash output from the intranet server. WordPress request traffic lives here. **Useful.**
- `internal-share` (1.1 GB) — logstash from the file-share host. Probably not useful for the attacks in this silo.
- `logs` (126 MB) — generic log folder. **Contains the attacker_0 attack timeline AND the inet-firewall dnsmasq logs.** The dnsmasq logs are where 44k of our labeled lines live. Critical.
- `audit` (4.7 MB) — auditd logs. Tiny but dense. User + privilege activity lives here. **Also critical.**
- `apache2`, `redis` — 0 MB. Empty. The WordPress activity is shipped via logstash, so it appears under `intranet-server`, not `apache2`. Ignore these two.

On the teacher's **no-feature-engineering** constraint: we're not going to hand-pick certain log types as "the important ones" and dump the rest into the model. But knowing which files are empty so we don't waste an hour writing an apache2 parser is not feature engineering — it's not stepping on a rake.

In [ ]:
rows = []
for host_dir in GATHER.iterdir():
    logs_dir = host_dir / "logs"
    if not logs_dir.is_dir():
        continue
    for log_file in logs_dir.rglob("*.log"):
        rel = log_file.relative_to(GATHER)
        rows.append({
            "host":     host_dir.name,
            "log_type": log_file.parent.name,
            "file":     log_file.name,
            "mb":       round(log_file.stat().st_size / 1e6, 2),
        })
inv = pd.DataFrame(rows)
print(inv.groupby("log_type")[["file","mb"]].agg({"file":"count","mb":"sum"}).sort_values("mb", ascending=False))

### raw log peek

Look at the actual format before writing any parser. Never assume. Every silo and log type has its own quirks, and one badly-parsed format wastes a whole day.

This is auditd. Format:

```
type=<TYPE> msg=audit(<epoch>.<ms>:<serial>): <key=value pairs>
```

Three things to notice:

1. **Line 1 is line 1.** Label files reference line *numbers*, not timestamps or IDs. If we sort this file, merge it with anything, or delete lines, every downstream label breaks. **Read it exactly as-is, count lines exactly.**
2. The epoch timestamp is inside `msg=audit(...)`. Parse with: `msg=audit\((\d+)\.\d+:`.
3. `type=USER_ACCT`, `type=USER_START`, `type=CRED_ACQ` — the audit event type is the first token on the line. That's a natural "token" for a sequence model: one log line = one event = one token. This is important for the LSTM path later — you don't have to invent a tokenizer, the log format gives you one.

In [ ]:
example = GATHER / "intranet_server" / "logs" / "audit" / "audit.log"
if not example.exists():
    example = next(GATHER.rglob("audit.log"))

with open(example) as f:
    for i, line in enumerate(f, start=1):
        if i > 5:
            break
        print(f"{i:>5}  {line.rstrip()[:180]}")

### labels peek

The label files are `.jsonl` — one JSON object per line. Each object has:

- `line`: which raw-line number it points to (1-indexed)
- `labels`: which attack steps this line belongs to (a line can have several — e.g. the same sudo command counts as both `escalate` and `attacker_change_user`)
- `rules`: which rule fired to assign the label (for debugging only, ignore)

Reading this record:

```json
{"line": 2079, "labels": ["attacker_change_user", "escalate"], ...}
```

means: *line 2079 of `gather/intranet_server/logs/audit/audit.log` is part of the privilege-escalation attack.*

The key mental model: **absence from a label file means normal.** There are no explicit "normal" labels. To build a binary y-vector you do:

```python
labeled = {json.loads(l)["line"] for l in open(label_file)}
y = [1 if i in labeled else 0 for i, _ in enumerate(open(raw_file), 1)]
```

And the **labels directory mirrors the gather directory**. `LABELS / "intranet_server/logs/audit/audit.log"` corresponds exactly to `GATHER / "intranet_server/logs/audit/audit.log"`. If your paths ever don't match on that, the join is broken.

In [ ]:
label_file = LABELS / example.relative_to(GATHER).with_suffix(".log")

with open(label_file) as f:
    for i, line in enumerate(f, start=1):
        if i > 3:
            break
        print(f"{line.rstrip()}")

### label taxonomy

What attack steps exist, and how often each appears.

**A clarification before the numbers.** The old version of this cell printed "Total labeled lines across all logs: 133,457". That is **not** the number of unique labeled lines — it's the sum of `(line, label)` pairs. A line with 3 labels adds 3. The code below now prints both, so you can see the actual unique count.

**Why the top three labels look almost identical:**

- `dnsteal` (44,018), `attacker` (44,018), `dnsteal-received` (44,016) — these are **co-labels on the same events**. Every exfil line gets all three. They are not three independent signals; they're one attack (DNS exfil) that got three tags.

**What's actually rare and would need care:**

- `webshell_cmd` (6), `escalated_sudo_session` (6), `escalated_sudo_command` (18) — post-exploit commands. Only a handful of lines. **Very hard to detect at the log-line level.**
- `escalate` (83), `crack_passwords` (46) — also rare. Hard.
- Scans: `service_scan` (90), `dns_scan` (414), `network_scan` (62), `wpscan` (8) — hundreds of events. **Detectable.**

**Takeaway:** this is not one task. "Detect dnsteal" is a totally different problem from "detect webshell_cmd". A model that nails 99% detection on dnsteal could completely miss all 6 webshell events and the averaged metric would look fine. Report **per-attack-type**, not averaged.

In [ ]:
all_labels = {}
for lf in LABELS.rglob("*.log"):
    with open(lf) as f:
        for raw in f:
            raw = raw.strip()
            if not raw:
                continue
            rec = json.loads(raw)
            for lbl in rec["labels"]:
                all_labels[lbl] = all_labels.get(lbl, 0) + 1

label_df = pd.DataFrame(
    sorted(all_labels.items(), key=lambda kv: -kv[1]),
    columns=["label", "count"],
)
print(label_df.to_string(index=False))

# Sum of (line, label) pairs — a line with 3 labels counts 3 times.
print(f"\nSum of (line, label) pairs: {sum(all_labels.values()):,}")

# Unique labeled lines — the real count.
unique_labeled = sum(
    1 for lf in LABELS.rglob("*.log")
    for line in open(lf)
    if line.strip()
)
print(f"Unique labeled lines:       {unique_labeled:,}")

### which log files actually contain attack signal

This is the cell that matters most for planning the model. For each raw log file, how many labeled lines does it contain?

For fox, the answer is:

- `inet-firewall/logs/dnsmasq.log` — **44,600** labeled lines. This is all the dnsteal data. **Exfil detector target.**
- `monitoring/logs/logstash/intranet-server/*` — 46. Attack steps appear here (logstash forwarding from the compromised host).
- `vpn/logs/*` — 28. The attacker's VPN connection events.
- `intranet_server/logs/audit/audit.log` — 19. The actual privesc / sudo events. **Privesc detector target.**
- `intranet_server/logs/*` (logstash) — 12. Post-exploit recon commands.
- `internal_share/logs/audit/audit.log` — 2. Minor.

**What's NOT here, and why it matters:**

- **`apache2` has zero labeled lines**, even though WordPress was compromised. That's because apache2 logs are empty in this silo (0 MB, from the inventory cell). The WordPress traffic is shipped to logstash, so it appears under `intranet-server`, not `apache2`. Grep-hunting in apache2 would find nothing.
- **`suricata` has zero labeled lines**, even though it's 2 GB. So the network-alert logs don't carry our attack signal. Skip Suricata unless you specifically want a network-level detector.

**Two detector targets emerge:**

1. **Exfil** → `inet-firewall/logs/dnsmasq.log`. 44k labeled lines. Easy to evaluate. But only catches exfil.
2. **Privesc / post-exploit** → `intranet_server/logs/audit/audit.log`. 19 labeled lines. Few samples. But this is the "interesting" attack.

This is the shape of the problem. Everything downstream is choosing how to model each of those two targets separately.

In [ ]:
per_log_type = []
for lf in LABELS.rglob("*.log"):
    log_type = lf.parent.name
    host = lf.relative_to(LABELS).parts[0]
    n = sum(1 for line in open(lf) if line.strip())
    if n:
        per_log_type.append({"host": host, "log_type": log_type, "n_labeled": n})

df = pd.DataFrame(per_log_type)
print("Total labeled lines by log_type:")
print(df.groupby("log_type")["n_labeled"].sum().sort_values(ascending=False))
print("\nTop 15 (host, log_type) pairs:")
print(df.sort_values("n_labeled", ascending=False).head(15).to_string(index=False))

### sanity check — do labels actually point at the right lines?

The whole join is "label files reference raw-line numbers". Before we trust that, spot-check it. Three random files is enough. We're not proving correctness — we're ruling out obvious bugs:

- labels dir doesn't mirror gather dir (path bug)
- raw log was sorted/modified somewhere (line numbers shifted)
- zip extraction was incomplete (line count mismatch)

The code picks one label file per log-type, reads the first label in it, and prints the referenced raw line. If the raw line looks like the attack it claims to be, we're good.

If all three look right, we assume the mechanism works everywhere. Testing 100 files is wasted effort — the code path is the same.

In [ ]:
def verify_alignment(sample_n=3):
    label_files = sorted(LABELS.rglob("*.log"))
    seen = set()
    samples = []
    for lf in label_files:
        lt = lf.parent.name
        if lt in seen:
            continue
        n = sum(1 for line in open(lf) if line.strip())
        if n:
            samples.append(lf)
            seen.add(lt)
        if len(samples) >= sample_n:
            break

    for lf in samples:
        raw_path = GATHER / lf.relative_to(LABELS)
        with open(raw_path) as f:
            raw_lines = f.readlines()

        first_rec = json.loads(next(open(lf)))
        ln = first_rec["line"]
        ok = 1 <= ln <= len(raw_lines)
        print(f"\n{lf.relative_to(LABELS)}")
        print(f"  labels: {first_rec['labels']}")
        print(f"  points to line {ln} of {len(raw_lines):,} in raw log  ->  {'OK' if ok else 'BROKEN'}")
        if ok:
            print(f"  raw line: {raw_lines[ln-1].rstrip()[:160]}")

verify_alignment()

### attack timeline (ground truth)

`attacker_0/logs/attacks.log` is the attacker's own step-by-step log — the ground-truth narrative of the attack. We read it once to understand the flow, and use it later to check whether our anomaly scores line up with what actually happened.

For fox, the flow is:

```
11:59  vpn_connect          attacker joins via VPN
11:59  dns_brute_force      scans DNS
11:59  host_discover_local  finds internal hosts
12:02  host_discover_dmz    finds DMZ hosts
12:17  service_scan         scans ports
12:18  dirb_scan            scans web paths
12:38  upload_rce_shell     <- webshell upload
12:38  check_user_id, check_netstat, ... (12 recon commands in 30 seconds)
12:38  crack_wphash         <- password cracking STARTS
13:13  reverse_shell_listen <- shell opens 35 min later
13:14  login_user, check_sudo, escalate via sudo
13:14  vpn_disconnect       attacker leaves
```

Three things to notice:

1. **There's a 35-minute gap from 12:38 to 13:13** where the attacker is idle (waiting for cracking to finish). Volume-based anomaly detectors will see nothing unusual here. But the attack is still running. **This is a fundamental limit** — either you use a long window (30+ min) that also dilutes normal behavior, or you accept missing the crack phase.

2. **The crack phase leaves essentially no log events.** John-the-ripper runs silently. You can't detect an attack that leaves no trace. Focus on the steps that DO log: scans, webshell upload, sudo commands.

3. **`dnsteal_stop` at 11:56** is odd — it's before `vpn_connect` at 11:59. Either leftover from a prior phase or the file isn't strictly sorted. Ignore that entry.

We don't train on this file. It's ground truth for the timeline plot below and for sanity-checking model outputs later.

In [ ]:
attacker_log = GATHER / "attacker_0" / "logs" / "attacks.log"
with open(attacker_log) as f:
    for line in f:
        print(line.rstrip()[:200])

### activity plot — what window size do we need?

Plot the labeled activity in `intranet_server/logs/audit/audit.log`, bucketed into 1-minute windows, over the attack window. The goal is to pick a feature-window size for later.

How to read the plot:

- **Sharp spikes** → attacks are event-dense and short. Use a small window (1-5 min).
- **Smeared activity** → attacks are spread out. Use a bigger window (10-15 min).
- **Long gaps between clusters** → no single window captures the whole attack. **Use a sequence model** (multiple windows per sample), not a single-window autoencoder.

For fox specifically:
- Audit activity concentrates around 12:38-12:39 (upload_rce_shell + recon commands).
- Another burst around 13:14 (privesc commands).
- About 35 min of silence between the two clusters.

**What this means for modelling:**

1. A 5-min window would see the 12:38 cluster but not the 13:14 privesc. To catch both, you need a **sequence** of windows (LSTM), not a single window.
2. For the **exfil** case (44k labeled lines in `dnsmasq.log`), it's continuous — shows up as elevated DNS query volume in any window that overlaps the attack. A 5-min windowed AE/VAE catches it easily.

**Conclusion:** different attacks need different window sizes. Don't try to pick one for everything. Exfil → 5-min windows. Multi-step privesc → sequence of 5-min windows feeding an LSTM.

In [ ]:
import re
from datetime import datetime

raw = GATHER / "intranet_server" / "logs" / "audit" / "audit.log"
lab = LABELS / "intranet_server" / "logs" / "audit" / "audit.log"

labeled_lines = set()
for raw_json in open(lab):
    if raw_json.strip():
        labeled_lines.add(json.loads(raw_json)["line"])

ts_re = re.compile(r"msg=audit\((\d+)\.\d+:")
times, labels = [], []
for i, line in enumerate(open(raw), start=1):
    m = ts_re.search(line)
    if not m:
        continue
    times.append(datetime.fromtimestamp(int(m.group(1))))
    labels.append(1 if i in labeled_lines else 0)

s = pd.Series(labels, index=pd.DatetimeIndex(times))
s = s.groupby(pd.Grouper(freq="1min")).sum()

win = s.loc["2022-01-18 11:30":"2022-01-18 13:30"]
win.plot(figsize=(12, 3), marker="o")
plt.title("Labeled audit-log lines per minute — intranet_server, fox")
plt.ylabel("labeled lines"); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

---

## Takeaways

What we now know, and what it means for the modelling step.

| question | answer (fox) | implication |
|---|---|---|
| simulation window | 2022-01-15 → 2022-01-20 | filter everything to this |
| hosts | 30, of which `attacker_0` must be excluded | exclude `attacker_0` from any normal-behavior training |
| biggest logs | suricata (2 GB), intranet-server (1.1 GB), logs (128 MB) | skip suricata, focus `logs` + `audit` |
| attack signal lives in | `inet-firewall/logs` (44k), `intranet_server/audit` (19) | two detector targets |
| exfil labels | 44k (`dnsteal` / `attacker` / `dnsteal-received` co-labeled) | easy to evaluate |
| privesc/webshell labels | 6-83 lines | hard to detect, sparse |
| label alignment | ✓ verified on 3 log types | trust the join |
| attack span | 11:59 → 13:14 (~75 min) | needs sequence models |

### What the no-feature-engineering constraint means for modelling

- **LSTM is the natural fit.** Feed raw log-event tokens (audit `type=` values, one token per event). The model learns the sequence. No manual feature crafting.
- **AE/VAE need fixed-size input.** Without feature engineering, you have two reasonable options: (a) bag-of-tokens per time window (counts of event types — tokenization, not really "engineering"), or (b) tokenize per event and feed through an embedding + pooling layer inside the autoencoder. Option (a) is simpler and probably what you want first.
- **Skip hand-crafted ratios** (failed_logins / total_logins, entropy of query names, etc.). Not allowed, and mostly not needed for the attacks in this silo.

### What to build next, in order

1. **Exfil detector** — windowed AE/VAE on `inet-firewall/logs/dnsmasq.log`, 5-min windows. 44k labeled lines to evaluate against. This will work. Use it to validate the full pipeline end-to-end (parse → window → train → score → evaluate).
2. **Privesc detector** — LSTM-AE on `intranet_server/logs/audit/audit.log` token sequences. Only 19 labeled lines to evaluate against, so expect noisy metrics. Report per-attack-type, not averaged.
3. **Compare AE vs VAE vs LSTM** — do it on the same silo, same evaluation protocol. The interesting question isn't "which wins on exfil" (they'll all win, data is generous) — it's "which wins on privesc" (data is sparse, sequence matters).

After that: repeat the pipeline on `harrison`, then `wheeler`, then wire up Flower for the federated part.